# A Memory the Agent Cannot Explain

Sarah Chen's travel assistant has a complete semantic-memory lifecycle. It can decide what is worth remembering, keep untrusted memories away from recommendations, replace outdated beliefs without deleting history, and retire low-value records.

Yet Sarah asks a simple question:

> **"Why do you think I prefer morning flights?"**

The graph contains more than the preference text. It also contains trust state, confidence, source, timestamps, and confirmation history. But does that information survive the trip from storage to the agent's response?

We will begin at the exact read boundary left by the lifecycle notebooks and inspect what the agent actually receives.

In [1]:
%pip install -q -r ../requirements.txt

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import json
import os
import sys

import certifi
import sniffio

sys.path.insert(0, ".")
sys.path.insert(0, "..")
sniffio.current_async_library_cvar.set("asyncio")
os.environ["SSL_CERT_FILE"] = certifi.where()

from dotenv import load_dotenv
from azure.ai.projects.aio import AIProjectClient as AsyncAIProjectClient
from azure.identity.aio import (
    AzureCliCredential as AsyncCliCredential,
    get_bearer_token_provider as async_get_bearer_token_provider,
)
from openai import AsyncAzureOpenAI
from agent_framework import Agent, AgentSession, tool
from lifecycle_utils import CosmosProvenanceStore
from shared.semantic_store import SemanticMemoryStore, create_container
from shared.travel_agent import (
    SYSTEM_PROMPT,
    create_client,
    get_travel_policy,
    search_flights,
    search_hotels,
)

load_dotenv("../.env", override=True)
client, credential = create_client("../.env")
print("Foundry client ready")

c:\Users\divyesheth\OneDrive - Microsoft\Documents\python-projects\agentic-memory\.venv\Lib\site-packages\agent_framework\_skills.py:122: ExperimentalWarning: [SKILLS] SkillResource is experimental and may change or be removed in future versions without notice.
c:\Users\divyesheth\OneDrive - Microsoft\Documents\python-projects\agentic-memory\.venv\Lib\site-packages\agent_framework\_harness\_file_access.py:602: ExperimentalWarning: [HARNESS] AgentFileStore is experimental and may change or be removed in future versions without notice.


Foundry client ready


In [ ]:
from azure.cosmos.aio import CosmosClient

embed_token = async_get_bearer_token_provider(
    AsyncCliCredential(),
    "https://cognitiveservices.azure.com/.default",
)
embed_deployment = os.environ.get(
    "AZURE_OPENAI_EMBEDDING_DEPLOYMENT", "text-embedding-ada-002"
)
embed_client = AsyncAzureOpenAI(
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
    azure_ad_token_provider=embed_token,
    api_version="2024-02-01",
)

async def embed(text: str) -> list[float]:
    r = await embed_client.embeddings.create(input=[text], model=embed_deployment)
    return r.data[0].embedding

cosmos = CosmosClient(os.environ["COSMOS_ENDPOINT"], credential=AsyncCliCredential())
container = await create_container(cosmos)
store = SemanticMemoryStore(container, user_id="E001", embed_fn=embed)
provenance = CosmosProvenanceStore(store)

module3_count = await provenance.prerequisite_count()
if module3_count == 0:
    raise RuntimeError(
        "No lifecycle memories found for E001. "
        "Run the staged-promotion and belief-revision notebooks first."
    )

await provenance.clear_provenance()
print(f"Found {module3_count} lifecycle memories for Sarah Chen (E001)")

Found 3 Module 03 lifecycle memories for Sarah Chen (E001)


## The Problem: What the Agent Actually Sees

The lifecycle notebooks left preferences in Neo4j with trust state, confidence, source type, and temporal history. But the read methods project only `[KNOWN]` or `[LIKELY]` — everything else is lost before it reaches the agent. Let's see that gap first-hand.

In [ ]:
@tool
async def recall_lifecycle_style(query: str) -> str:
    """Recall beliefs using only lifecycle state labels — no evidence or confidence."""
    return await provenance.recall_baseline(query)

baseline_agent = Agent(
    client=client,
    name="LifecycleOnlyTravelAssistant",
    instructions=SYSTEM_PROMPT + "\n\n" + (
        "Before describing the user's preferences, call recall_lifecycle_style. "
        "Use only what that tool returns. If asked why a belief exists, do not "
        "invent a source or supporting evidence that the tool did not provide."
    ),
    tools=[recall_lifecycle_style],
)

baseline_session = AgentSession()
result = await baseline_agent.run(
    "What travel preferences do you remember for me? And why do you think I prefer those things?",
    session=baseline_session,
)
print("ASSISTANT:", result.text)

=== Existing Module 03 Recall ===


Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. db.index.vector.queryNodes is deprecated. It is replaced by SEARCH.', position=<SummaryInputPosition line=2, column=1, offset=1>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 1, 'line': 2, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "\nCALL db.index.vector.queryNodes('preference_embedding_idx', $limit, $embedding)\nYIELD node, score\nWHERE score >= $threshold\nRETURN node AS p, score\nORDER BY score DESC\n"


[LIKELY] Paris

=== Existing Nodes Before Provenance Enrichment ===
[provisional] home_city: New York | evidence=None
[provisional] home_airport: JFK | evidence=None
[provisional] home_city: Paris | evidence=None


## What Went Wrong

The data in Neo4j is fine — the read methods intentionally projected only the fields each lifecycle lesson needed.

| Lifecycle capability | Question it answers | Question still unanswered |
|---|---|---|
| Identification | Should this be remembered? | Why was this particular memory created? |
| Staged promotion | May it influence the agent? | How strong was the supporting evidence? |
| Belief revision | Which value is current? | What caused each version to exist? |
| Retention | Is it worth keeping? | How should the agent communicate its certainty? |

Two records can both be labelled `[LIKELY]`, even though one is a direct user statement at `0.95` confidence and the other is a weak inference at `0.55`. The baseline output contains no evidence with which to answer "why?". Asking the model to fill that gap produces a generated rationale, not an audit trail.

## The Missing Concept: Provenance

**Provenance is the traceable origin and evidence chain of a memory.** It should let the system answer:

1. **Who or what created it?** The user, an agent inference, a tool, or an enterprise source.
2. **Which interaction or evidence supported it?** The actual statement or observation—not merely a generic source label.
3. **When was it recorded and confirmed?** Timestamps place the evidence in context.
4. **How was it validated?** Trust state and confirmation count show what happened after creation.
5. **How did it change?** Lineage connects the current value to superseded versions.

`source_type="llm_inference"` alone is not enough. It identifies the kind of origin, but it does not explain the inference. `source_detail="Three recent booked trips departed before 9:00 AM"` provides inspectable evidence.

### Provenance is not confidence

- **Provenance** says *where and why* a memory came from.
- **Confidence** estimates *how strongly* the system should believe and communicate it.
- **Temporal history** says *how the value changed*.
- **Trust state** says *whether it may influence the agent now*.

Persisting provenance matters because a model can otherwise invent a convincing explanation after the fact. Stored evidence lets users inspect and correct memories, lets developers debug bad personalization, and makes the memory an auditable artifact rather than an unexplained assertion.

## The Solution: Preserve Metadata Through the Read Boundary

`GraphProvenanceStore` keeps the lifecycle rules but returns structured provenance instead of reducing a node to text too early.

```mermaid
flowchart LR
    A[Neo4j Preference node] --> B{Lifecycle state}
    B -->|candidate / deprecated| C[Withhold]
    B -->|provisional / trusted| D[Structured recall]
    D --> E{State + confidence}
    E --> F[Assert]
    E --> G[Hedge]
    E --> H[Ask]
    A --> I[Source + evidence + actor + time]
    I --> J[Explain belief]
```

The order is important: **state controls visibility first; confidence controls wording second.**

| State | Confidence | Behavior |
|---|---:|---|
| `candidate` or `deprecated` | any | Withhold |
| `provisional` | below `0.5` | Ask |
| `provisional` | `0.5` or above | Hedge |
| `trusted` | `0.8` or above | Assert |
| `trusted` | `0.5`–`0.79` | Hedge |
| `trusted` | below `0.5` | Ask |

A high confidence score cannot bypass quarantine, and a provisional memory is never presented as settled fact.

In [ ]:
@tool
async def recall_beliefs(query: str) -> str:
    """Recall current beliefs with mandatory assert, hedge, or ask behaviour."""
    rows = await provenance.recall_with_confidence(query)
    if not rows:
        return "No current beliefs are eligible for recall."
    return json.dumps(rows, indent=2, default=str)


@tool
async def confirm_belief(preference: str) -> str:
    """Confirm or reaffirm a belief the user has validated. Advances trust state."""
    result = await provenance.confirm(preference)
    if result.get("error"):
        return f"Could not confirm: {result['error']}"
    return (
        f"Confirmed '{result['preference']}' → "
        f"state={result['state']}, confirmations={result['confirmations']}"
    )


@tool
async def explain_belief(category: str) -> str:
    """Explain why the current belief in a category exists, using stored evidence only."""
    record = await provenance.explain(category)
    if record is None:
        return f"No current belief exists for category '{category}'."

    first_seen = record["first_seen"][:10] if record.get("first_seen") else "unknown"
    last_confirmed = (
        record["last_confirmed"][:10]
        if record.get("last_confirmed")
        else "not yet confirmed"
    )
    lineage = " -> ".join(item["preference"] for item in record["history"])
    evidence = record.get("source_detail") or "No supporting evidence was stored."
    creator = record.get("created_by") or "unknown creator"
    source_type = record.get("source_type") or "unknown source"
    confidence = record.get("confidence") or 0.0
    confirmations = record.get("confirmation_count") or 0
    return (
        f"Belief: {record['preference']}\n"
        f"Origin: {source_type} via {creator}\n"
        f"Evidence: {evidence}\n"
        f"First recorded: {first_seen}\n"
        f"Confidence: {confidence:.2f}\n"
        f"State: {record['state']} ({confirmations} confirmations; "
        f"last confirmed: {last_confirmed})\n"
        f"Lineage: {lineage}"
    )


print("Tools ready: recall_beliefs, confirm_belief, explain_belief")

Provenance-aware read tools ready


In [ ]:
assistant = Agent(
    client=client,
    name="ProvenanceAwareTravelAssistant",
    instructions=SYSTEM_PROMPT + "\n\n" + (
        "You have access to Sarah Chen's long-term memory.\n"
        "- Before personalizing a response, call recall_beliefs. Obey every presentation "
        "field exactly: assert as fact, hedge as uncertain and seek confirmation, or ask "
        "a direct clarification question. Never use a belief absent from recall.\n"
        "- When the user explicitly confirms or reaffirms a preference, call confirm_belief "
        "so the system can advance its trust state.\n"
        "- For questions containing 'why', 'how do you know', or 'where did that come "
        "from', call explain_belief. Use only its stored evidence; never invent provenance."
    ),
    tools=[
        search_flights,
        search_hotels,
        get_travel_policy,
        recall_beliefs,
        confirm_belief,
        explain_belief,
    ],
)
print(f"Agent ready: {assistant.name}")

Agent ready: ProvenanceAwareTravelAssistant


## Enriching Existing Memories with Evidence

The preferences below were created by the lifecycle notebooks. We now attach the evidence from those earlier interactions to the **same Neo4j nodes** — no new preferences are inserted. In a production system, evidence would be captured at write time; here we backfill what the earlier notebooks didn't persist.

In [ ]:
await provenance.enrich(
    category="home_city",
    source_detail="Sarah said: 'I just moved to Paris permanently.'",
    created_by="Sarah Chen via TravelAssistant",
)
await provenance.enrich(
    category="hotel_chain",
    source_detail="Sarah said: 'I always stay at Marriott when travelling.'",
    created_by="Sarah Chen via TravelAssistant",
)

count_after = await provenance.prerequisite_count()
assert count_after == module3_count, "Enrichment must not create new preferences"
print(f"Enriched existing memories ({count_after} preferences unchanged)")

Hotel enrichment: {
  "error": "module_03_memory_not_found",
  "category": "hotel_chain"
}

Home-city enrichment: {
  "id": "1126aa46-4d6a-46b0-8dae-fce7c56ba48d",
  "preference": "Paris",
  "category": "home_city",
  "state": "provisional",
  "confidence": 0.8,
  "source_type": "user_assertion",
  "source_detail": "In Module 3.4 Sarah said: 'I just moved to Paris permanently.'",
  "created_by": "Sarah Chen via TravelAssistant"
}

Existing memories after enrichment:
[
  {
    "preference": "New York",
    "category": "home_city",
    "state": "provisional",
    "confidence": 0.8,
    "source_type": "user_assertion",
    "source_detail": null,
    "created_by": null,
    "confirmation_count": 0,
    "first_seen": "2026-09-07T12:36:18.334Z",
    "valid_from": "2026-09-07T12:36:18.334Z",
    "valid_to": "2026-09-07T12:36:40.731Z"
  },
  {
    "preference": "JFK",
    "category": "home_airport",
    "state": "provisional",
    "confidence": 0.8,
    "source_type": "user_assertion",
    "so

## The Payoff: Every Presentation Behaviour in One Conversation

The same agent, the same memories, one continuous session. Watch the agent's language change as the underlying trust state evolves:

| Turn | What happens | Expected behaviour |
|------|-------------|-------------------|
| 1 | Recall preferences | **Hedge** — everything starts provisional |
| 2 | User confirms Paris | **Confirm** → state advances to trusted |
| 3 | Recall again | **Assert** — Paris is now stated as fact |
| 4 | Ask "why Paris?" | **Explain** — cites stored evidence + lineage |
| 5 | Ask about missing preference | **Ask** — no data, so agent asks user |

In [ ]:
session = AgentSession()

# Turn 1 — agent recalls provisional beliefs and must hedge
r1 = await assistant.run(
    "What travel preferences do you have on file for me?",
    session=session,
)
print("ASSISTANT:", r1.text)

In [ ]:
# Turn 2 — user confirms Paris; agent should call confirm_belief → trusted
r2 = await assistant.run(
    "Yes, I definitely live in Paris now — that's confirmed.",
    session=session,
)
print("ASSISTANT:", r2.text)

In [ ]:
# Turn 3 — same question, but Paris is now trusted → agent should ASSERT
r3 = await assistant.run(
    "So where should I be flying from for my next trip?",
    session=session,
)
print("ASSISTANT:", r3.text)

### The Same Memory, Two Behaviours

Compare Turn 1 and Turn 3. The underlying preference ("Paris") hasn't changed — but the agent's language shifted from hedging to asserting because the trust state advanced from `provisional` to `trusted` after the user's confirmation in Turn 2.

In [ ]:
# Turn 4 — "why?" → agent calls explain_belief, cites stored evidence + lineage
r4 = await assistant.run(
    "Why do you think I live in Paris? Where did that come from?",
    session=session,
)
print("ASSISTANT:", r4.text)

In [ ]:
# Turn 5 — ask about something without stored data → agent must ask, not guess
r5 = await assistant.run(
    "Do you know what seat I prefer on long flights?",
    session=session,
)
print("ASSISTANT:", r5.text)

### Missing Provenance Must Stay Missing

If evidence was not persisted, the agent must report that no supporting evidence exists rather than inventing a plausible rationale. The next turn asks about a preference whose enrichment failed (no matching node) — the agent should say so honestly.

In [ ]:
# Turn 6 — explain a preference that may lack evidence → honest gap
r6 = await assistant.run(
    "Why do you think I prefer Marriott? Where did that come from?",
    session=session,
)
print("ASSISTANT:", r6.text)

## Key Takeaways

1. **This notebook builds directly on the lifecycle notebooks.** It reads and enriches the memories already present in Neo4j; it does not create a parallel dataset.
2. **Provenance is an evidence chain, not a source label.** Preserve the actor, source type, supporting interaction or observation, timestamps, confirmations, and lineage.
3. **State comes before confidence.** Candidates remain hidden regardless of score; provisional beliefs never become facts merely because their numeric confidence is high.
4. **Confidence changes language.** The same memory can require an assertion, a hedge, or a clarification question.
5. **Confirmation changes behaviour.** A single user reaffirmation promoted Paris from hedged to asserted — the same data, a different trust state, a different response.
6. **Explanations must be retrieved, not improvised.** If evidence was not persisted, the agent must say it is unavailable.
7. **One graph remains the source of truth.** Provenance is metadata on the existing preference nodes; `GraphProvenanceStore` keeps this lesson's Python interface focused.

## Next: Retrieval and Routing

Now that recalled memories retain trust, confidence, and provenance, the next step decides which memory system should answer a request and how the most relevant records reach the agent.

In [ ]:
final_count = await provenance.prerequisite_count()
assert final_count == module3_count, "No preferences should have been created or deleted"
print(f"Preference count unchanged: {final_count}")

await cosmos.close()
print("Complete — lifecycle memories and provenance remain in Cosmos DB")

Module 04 complete; Module 03 memories and provenance remain in Neo4j
